In [8]:
from pathlib import Path
import hashlib, json, re, os, time, gc, importlib.util, urllib.request
import pandas as pd

# 1. Tùy chỉnh chạy (giữ 01 run để tương thích Qwen đã thực nghiệm).
RUN_IDS = [1]
RUN_MODELS = ["Qwen2.5-Coder-3B", "NSQL-350M", "SQLCoder-7B-2", "NSQL-Llama-2-7B"]
TEST_LIMIT = None                     # None: đủ 43 câu/model; 2: thử 2 câu
RUN_INFERENCE = True                  # Đặt False nếu chỉ kiểm tra prompt / đánh giá
# 7B: GGUF Q4_K_M qua Ollama để không bắt buộc GPU >=16 GiB.
# Mặc định offload một số lớp lên T1000 4 GiB, còn lại chạy trên RAM/CPU.
AUTO_PULL_7B = True                   # Tự tải GGUF nếu Ollama chưa có (lần đầu cần Internet + ~10GB ổ đĩa)
OLLAMA_7B_GPU_LAYERS = 8             # 0 = chỉ CPU (an toàn nếu OOM); 8 = offload một phần lên GPU
OLLAMA_7B_KEEP_ALIVE = "5m"           # Sau khi chạy xong Ollama có thể giải phóng RAM/VRAM
USE_4BIT_7B = False                   # Cờ cũ không cần cho GGUF; giữ để tương thích notebook
STOP_ON_MODEL_ERROR = False           # Lỗi model không ngăn model kế tiếp
REUSE_LEGACY_QWEN = True              # Giữ nguyên CSV Qwen cũ, không tự ghi đè
OLLAMA_URL = "http://127.0.0.1:11434"

MODEL_REGISTRY = {
    "Qwen2.5-Coder-3B": {"model_id": "qwen2.5-coder:3b-instruct", "backend": "ollama",
                           "slug": "qwen2_5_coder_3b", "schema": "full", "context": 8192, "max_new": 1024},
    "NSQL-350M": {"model_id": "NumbersStation/nsql-350M", "backend": "transformers",
                   "slug": "nsql_350m", "schema": "compact", "context": 2048, "max_new": 256},
    "SQLCoder-7B-2": {"model_id": "defog/sqlcoder-7b-2", "backend": "ollama_gguf",
                       "slug": "sqlcoder_7b_2", "schema": "full", "context": 8192, "max_new": 512,
                         "serving_id": "hf.co/QuantFactory/sqlcoder-7b-2-GGUF:Q4_K_M", "quantization": "Q4_K_M", "raw_prompt": True},
    "NSQL-Llama-2-7B": {"model_id": "NumbersStation/nsql-llama-2-7B", "backend": "ollama_gguf",
                         "slug": "nsql_llama_2_7b", "schema": "full", "context": 4096, "max_new": 512,
                         "serving_id": "hf.co/mradermacher/nsql-llama-2-7B-GGUF:Q4_K_M", "quantization": "Q4_K_M", "raw_prompt": True},
}
assert all(x in MODEL_REGISTRY for x in RUN_MODELS)
assert RUN_IDS and len(RUN_IDS) == len(set(RUN_IDS)) and all(isinstance(i, int) and i > 0 for i in RUN_IDS)

# 2. Định vị project bằng dữ liệu đã tạo; không đoán đường dẫn tuyệt đối của Windows.
PROJECT_DIR = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "results/llm/reproduction/generation_run_01.csv").is_file()), None)
assert PROJECT_DIR is not None, "Không thấy project. Mở notebook từ thư mục con của thuc-nghiem-co-so-httt."
BASELINE_FILE = PROJECT_DIR / "results/llm/reproduction/generation_run_01.csv"
DDL_FILE = PROJECT_DIR / "data/ACME_Insurance/DDL/ACME_small.ddl"
DB_FILE = PROJECT_DIR / "results/sql/acme.duckdb"
REF_DIR = PROJECT_DIR / "results/sql/reference_results"
REF_MAP = PROJECT_DIR / "results/ontop/reference_comparison.csv"
EVAL_NOTEBOOK = PROJECT_DIR / "notebook/05_evaluation.ipynb"
QUADRANT_FILE = PROJECT_DIR / "results/evaluation/oea_by_quadrant.csv"
OUT_DIR = PROJECT_DIR / "results/related_methods/model_baselines"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in (BASELINE_FILE, DDL_FILE):
    assert p.is_file(), f"Thiếu file dữ liệu gốc: {p}"

# 3. Lấy đúng 43 câu SQL, không thay đổi benchmark.
baseline = pd.read_csv(BASELINE_FILE, keep_default_na=False)
required = {"ID", "Method", "Inquiry URI", "Question"}
assert required.issubset(baseline.columns), f"Thiếu cột: {required - set(baseline.columns)}"
questions = baseline.loc[baseline["Method"].eq("SQL"), ["ID", "Inquiry URI", "Question"]].copy()
questions["ID"] = questions["ID"].astype(int)
questions = questions.sort_values("ID").reset_index(drop=True)
assert len(questions) == 43 and questions["ID"].nunique() == 43
assert questions["Inquiry URI"].nunique() == 43 and questions["Question"].astype(str).str.strip().ne("").all()
ddl_text = DDL_FILE.read_text(encoding="utf-8-sig")
assert ddl_text.strip()
questions.to_csv(OUT_DIR / "questions.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{"Model": n, **v} for n, v in MODEL_REGISTRY.items()]).to_csv(
    OUT_DIR / "model_registry.csv", index=False, encoding="utf-8-sig")
print(f"Project: {PROJECT_DIR}\nQuestions: {len(questions)} | Models: {len(RUN_MODELS)} | Runs: {RUN_IDS}")
print("DDL chars:", len(ddl_text), "| Output:", OUT_DIR)
display(questions.head(3))

Project: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Questions: 43 | Models: 4 | Runs: [1]
DDL chars: 6154 | Output: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\related_methods\model_baselines


,ID,Inquiry URI,Question
0,1,https://templates.data.world/IQ_03fc5a5571db11...,What is the loss ratio of each policy and agen...
1,2,https://templates.data.world/IQ_0d87a2a016062b...,Return all the policies we have by policy numb...
2,3,https://templates.data.world/IQ_1e19282f42eaea...,"What are the loss payment, loss reserve, expen..."


In [9]:
# 1. Tách đúng từng CREATE TABLE bằng dấu ngoặc cân bằng (VARCHAR(n), DECIMAL(p,s) không bị cắt).
def split_top_level(text, sep=","):
    parts, buf, depth, quote = [], [], 0, None
    for ch in text:
        if quote:
            buf.append(ch)
            if ch == quote:
                quote = None
        elif ch in ("'", '"', "`"):
            quote = ch; buf.append(ch)
        elif ch == "(":
            depth += 1; buf.append(ch)
        elif ch == ")":
            depth -= 1; buf.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(buf).strip()); buf = []
        else:
            buf.append(ch)
    if buf: parts.append("".join(buf).strip())
    assert depth == 0, "DDL có dấu ngoặc không cân bằng"
    return [p for p in parts if p]

def parse_tables(ddl):
    ddl = re.sub(r"/\*.*?\*/", "", ddl, flags=re.S)
    ddl = re.sub(r"--[^\n]*", "", ddl)
    tables = []
    pattern = re.compile(r"\bCREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?([\w.\"`]+)\s*\(", re.I)
    for m in pattern.finditer(ddl):
        start, depth, quote, end = m.end(), 1, None, None
        for pos in range(start, len(ddl)):
            ch = ddl[pos]
            if quote:
                if ch == quote: quote = None
            elif ch in ("'", '"', "`"):
                quote = ch
            elif ch == "(": depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0:
                    end = pos; break
        assert end is not None, f"CREATE TABLE không đóng ngoặc: {m.group(1)}"
        tables.append((m.group(1), split_top_level(ddl[start:end])))
    assert tables and len({name.lower() for name, _ in tables}) == len(tables), "Không tìm thấy bảng hoặc bảng bị trùng"
    return tables

def compact_ddl(ddl):
    """Giữ 100% tên bảng/cột/PK/FK; chỉ bỏ NULL/NOT NULL, tên constraint và độ dài VARCHAR."""
    tables = parse_tables(ddl)
    output = []
    for name, clauses in tables:
        compact_clauses = []
        for clause in clauses:
            c = " ".join(clause.split())
            c = re.sub(r"^CONSTRAINT\s+[\w\"`]+\s+", "", c, flags=re.I)
            # Chỉ bỏ nullability của cột, không được sửa "ON DELETE SET NULL" trong khóa ngoại.
            if not re.match(r"^(?:PRIMARY\s+KEY|FOREIGN\s+KEY|UNIQUE|CHECK)\b", c, re.I):
                c = re.sub(r"\s+(?:NOT\s+)?NULL\s*$", "", c, flags=re.I)
            c = re.sub(r"\b(N?VARCHAR|N?CHAR|CHARACTER\s+VARYING)\s*\(\s*\d+\s*\)",
                       lambda m: m.group(1), c, flags=re.I)
            c = re.sub(r"\s+", " ", c).strip()
            c = re.sub(r"\s*([(),])\s*", r"\1", c)
            compact_clauses.append(c)
        assert len(compact_clauses) == len(clauses), "Không được xóa cột/constraint"
        output.append(f"CREATE TABLE {name}(" + ",".join(compact_clauses) + ");")
    return "\n".join(output), tables

compact_text, ddl_tables = compact_ddl(ddl_text)
assert len(ddl_tables) == 13, f"DDL khác nguồn đã kiểm tra (13 bảng): got {len(ddl_tables)}"
assert len(parse_tables(compact_text)) == len(ddl_tables)
for (original_name, original_parts), (new_name, new_parts) in zip(ddl_tables, parse_tables(compact_text)):
    assert original_name == new_name and len(original_parts) == len(new_parts)
    def identifiers(parts):
        return [p.strip().split()[0].strip('"`').lower() for p in parts
                if not re.match(r"^(?:CONSTRAINT\s+\S+\s+)?(?:PRIMARY|FOREIGN|UNIQUE|CHECK)\b", p.strip(), re.I)]
    assert identifiers(original_parts) == identifiers(new_parts), f"Mất cột của bảng {original_name}"

# 2. Template riêng của từng tác giả. Qwen giữ nguyên prompt cũ để tái sử dụng checkpoint.
def build_model_prompt(model_name, question, ddl):
    if model_name == "Qwen2.5-Coder-3B":
        return ("Given the database described by the following DDL:\n" + ddl + "\n"
                "Write a SQL query that answers the following question. "
                "Do not explain the query. return just the query, "
                "so it can be run verbatim from your response.\n"
                "Here’s the question:\n" + f'"{question}"')
    if model_name.startswith("NSQL"):
        return (ddl.strip() + "\n\n"
                "-- Using valid DuckDB SQL, answer the following question for the tables provided above.\n"
                f"-- {question}\n\nSELECT")
    if model_name == "SQLCoder-7B-2":
        return ("### Task\n" + f"Generate a SQL query to answer [QUESTION]{question}[/QUESTION]\n\n"
                "### Database Schema\nThe query will run on a database with the following schema:\n"
                f"{ddl}\n\n### Answer\nGiven the database schema, here is the SQL query that "
                f"[QUESTION]{question}[/QUESTION]\n[SQL]\n")
    raise ValueError(model_name)

records = []
for model_name in RUN_MODELS:
    spec = MODEL_REGISTRY[model_name]
    source_schema = compact_text if spec["schema"] == "compact" else ddl_text
    for r in questions.itertuples(index=False):
        prompt = build_model_prompt(model_name, r.Question, source_schema)
        records.append({"Model": model_name, "Model ID": spec["model_id"], "ID": int(r.ID),
                        "Inquiry URI": getattr(r, '_1', None) or r[1], "Question": r.Question,
                        "Schema Variant": spec["schema"], "Prompt": prompt,
                        "Prompt Characters": len(prompt),
                        "Prompt SHA256": hashlib.sha256(prompt.encode("utf-8")).hexdigest()})
model_prompts = pd.DataFrame(records)
assert len(model_prompts) == 43 * len(RUN_MODELS) and not model_prompts.duplicated(["Model", "ID"]).any()
assert model_prompts.loc[model_prompts["Model"].str.startswith("NSQL"), "Prompt"].str.endswith("SELECT").all()
model_prompts.to_csv(OUT_DIR / "model_prompts.csv", index=False, encoding="utf-8-sig")
(OUT_DIR / "schema_compact.sql").write_text(compact_text, encoding="utf-8")
print(f"DDL {len(ddl_tables)} tables: full {len(ddl_text)} chars → compact {len(compact_text)} chars")
display(model_prompts.groupby(["Model", "Schema Variant"])["Prompt Characters"].agg(["min", "max"]))
print("Saved model_prompts.csv, schema_compact.sql (không đụng vào DB)")

DDL 13 tables: full 6154 chars → compact 5033 chars


,,min,max
Model,Schema Variant,,
NSQL-350M,compact,5161,5364
NSQL-Llama-2-7B,full,6281,6484
Qwen2.5-Coder-3B,full,6406,6609
SQLCoder-7B-2,full,6446,6852


Saved model_prompts.csv, schema_compact.sql (không đụng vào DB)


In [10]:
import sys
print("=== PREFLIGHT ===")
print("Python:", sys.version.split()[0])
for package in ["pandas", "duckdb", "torch", "transformers", "accelerate", "safetensors", "sentencepiece"]:
    print(f"{package:14s}", "OK" if importlib.util.find_spec(package) else "MISSING")

if importlib.util.find_spec("torch"):
    import torch
    print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0),
              "| Total VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GiB")

# 1. Audit token bằng tokenizer đúng của từng HF model; không chỉ tính số ký tự.
TOKENIZERS = {}
token_rows = []
if importlib.util.find_spec("transformers"):
    from transformers import AutoTokenizer
    for name in RUN_MODELS:
        cfg = MODEL_REGISTRY[name]
        if cfg["backend"] not in ("transformers", "ollama_gguf"): continue
        print("Tokenizer:", name, flush=True)
        try:
            tok = AutoTokenizer.from_pretrained(cfg["model_id"], trust_remote_code=False)
            TOKENIZERS[name] = tok
            for row in model_prompts[model_prompts["Model"].eq(name)].itertuples(index=False):
                prompt = row[model_prompts.columns.get_loc("Prompt")]
                tokens = len(tok.encode(prompt, add_special_tokens=False))
                token_rows.append({"Model": name, "ID": int(row.ID), "Prompt Tokens": tokens,
                                   "Reserved Output": cfg["max_new"], "Max Context": cfg["context"],
                                   "Fits Context": tokens + cfg["max_new"] <= cfg["context"]})
        except Exception as e:
            print(f"Cannot audit {name}: {type(e).__name__}: {str(e)[:180]}")
else:
    print("Chưa có transformers — HF inference sẽ bị chặn (không sử dụng số token ước lượng).")

TOKEN_AUDIT = pd.DataFrame(token_rows)
if not TOKEN_AUDIT.empty:
    TOKEN_AUDIT.to_csv(OUT_DIR / "token_audit.csv", index=False, encoding="utf-8-sig")
    display(TOKEN_AUDIT.groupby("Model").agg(
        min_tokens=("Prompt Tokens", "min"), max_tokens=("Prompt Tokens", "max"),
        fits=("Fits Context", "sum"), total=("ID", "count")))
    for name in RUN_MODELS:
        if MODEL_REGISTRY[name]["backend"] not in ("transformers", "ollama_gguf"): continue
        rows = TOKEN_AUDIT[TOKEN_AUDIT["Model"].eq(name)]
        if len(rows) == 43 and not rows["Fits Context"].all():
            print(f"BLOCKED: {name} có {len(rows) - int(rows['Fits Context'].sum())} prompt vượt context")
else:
    print("Token audit chưa thực hiện được.")

# 2. Kiểm tra Qwen Ollama mà không tải weights ở bước kiểm thử.
try:
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as resp:
        ollama_tags = {m["name"] for m in json.load(resp).get("models", [])}
    print("Ollama models:", sorted(ollama_tags))
except Exception as e:
    print("Ollama unavailable:", str(e)[:130])
print("GGUF models: SQLCoder-7B-2, NSQL-Llama-2-7B sử dụng Ollama Q4_K_M + CPU/RAM offload")
print("Preflight complete. Cell 4 mới tiến hành inference.")

=== PREFLIGHT ===
Python: 3.12.10
pandas         OK
duckdb         OK
torch          OK
transformers   OK
accelerate     OK
safetensors    OK
sentencepiece  OK
PyTorch: 2.15.0+cu132 | CUDA: True
GPU: Quadro T1000 | Total VRAM: 4.0 GiB
Tokenizer: NSQL-350M


Tokenizer: SQLCoder-7B-2
Tokenizer: NSQL-Llama-2-7B


,min_tokens,max_tokens,fits,total
Model,,,,
NSQL-350M,1581,1621,43,43
NSQL-Llama-2-7B,2098,2142,43,43
SQLCoder-7B-2,2153,2239,43,43


Ollama models: ['qwen2.5-coder:3b-instruct', 'qwen3.5:4b', 'qwen3.5:9b']
GGUF models: SQLCoder-7B-2, NSQL-Llama-2-7B sử dụng Ollama Q4_K_M + CPU/RAM offload
Preflight complete. Cell 4 mới tiến hành inference.


In [11]:
import urllib.error
import subprocess, shutil

RESULT_COLUMNS = ["ID", "Inquiry URI", "Question", "Method", "Model", "Run", "Model ID",
                  "Backend", "Schema Variant", "Prompt SHA256", "Generated Query",
                  "Prompt Tokens", "Output Tokens", "Time (s)", "Status", "Error", "Serving Model", "Quantization"]

def result_path(name, run_id):
    return OUT_DIR / f"{MODEL_REGISTRY[name]['slug']}_run_{run_id:02d}.csv"

def save_checkpoint(path, records):
    df = pd.DataFrame(records, columns=RESULT_COLUMNS).sort_values("ID")
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    os.replace(tmp, path)

def current_ollama_models():
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=15) as resp:
        return {str(m.get("name", "")).casefold() for m in json.load(resp).get("models", [])}


def ensure_ollama_model(cfg):
    """Chỉ auto-pull model 7B qua Ollama; Qwen legacy giữ nguyên, không tự thay checkpoint."""
    serving_id = cfg.get("serving_id", cfg["model_id"])
    if serving_id.casefold() in current_ollama_models():
        print("Ollama model present:", serving_id, flush=True)
        return serving_id
    if not cfg.get("serving_id"):
        raise RuntimeError(f"Ollama chưa có model {serving_id}; chạy: ollama pull {serving_id}")
    if not AUTO_PULL_7B:
        raise RuntimeError(f"Chưa có GGUF; chạy trong PowerShell: ollama pull {serving_id}")
    if not shutil.which("ollama"):
        raise RuntimeError(
            f"Ollama service có thể đang chạy nhưng không tìm thấy lệnh ollama trong PATH. "
            f"Mở PowerShell chạy: ollama pull {serving_id}; sau đó chạy lại Cell 4.")
    print("Downloading GGUF via Ollama (first use, ~4-5GB per model):", serving_id, flush=True)
    proc = subprocess.Popen(["ollama", "pull", serving_id], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, errors="replace")
    # Tránh giữ trong bộ nhớ toàn bộ progress khi stdout sử dụng carriage-return.
    recent = []
    for line in proc.stdout:
        cleaned = line.strip().replace("\r", " ")
        if cleaned:
            recent.append(cleaned[-300:]); recent = recent[-5:]
            if "success" in cleaned.lower() or "error" in cleaned.lower():
                print(cleaned[-300:], flush=True)
    ret = proc.wait()
    if ret:
        raise RuntimeError(f"ollama pull failed ({ret}) {serving_id}: {'; '.join(recent)}")
    if serving_id.casefold() not in current_ollama_models():
        raise RuntimeError("Download finished but /api/tags does not show: " + serving_id)
    print("Downloaded:", serving_id, flush=True)
    return serving_id


def ollama_generate(prompt, cfg):
    serving_id = cfg.get("serving_id", cfg["model_id"])
    options = {"temperature": 0, "num_predict": cfg["max_new"],
               "num_ctx": cfg["context"], "seed": 7}
    if cfg.get("raw_prompt"):
        options["num_gpu"] = int(OLLAMA_7B_GPU_LAYERS)
    request_obj = {"model": serving_id, "prompt": prompt, "stream": False,
                   "options": options}
    if cfg.get("raw_prompt"):
        # Giữ nguyên các prompt [SQL] và SELECT thay vì Ollama tự thêm chat template.
        request_obj["raw"] = True
        request_obj["keep_alive"] = OLLAMA_7B_KEEP_ALIVE
    def call(data):
        request = urllib.request.Request(OLLAMA_URL + "/api/generate",
                    data=json.dumps(data).encode("utf-8"),
                    headers={"Content-Type": "application/json"}, method="POST")
        try:
            with urllib.request.urlopen(request, timeout=3600 if cfg.get("raw_prompt") else 600) as response:
                obj = json.load(response)
        except urllib.error.HTTPError as e:
            # Ollama đôi khi trả mã 500 cùng thông báo thiếu bộ nhớ trong HTTP body.
            body = e.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"Ollama HTTP {e.code}: {body[:900]}") from e
        if obj.get("error"):
            raise RuntimeError(str(obj["error"]))
        return str(obj.get("response", "")).strip(), obj.get("prompt_eval_count", ""), obj.get("eval_count", "")
    try:
        return call(request_obj)
    except Exception as e:
        error_message = str(e).lower()
        if cfg.get("raw_prompt") and options["num_gpu"] > 0 and any(
                v in error_message for v in ["out of memory", "not enough memory", "cuda", "gpu memory"]):
            print("GPU offload failed, retrying model on CPU/RAM:", e, flush=True)
            request_obj["options"] = {**options, "num_gpu": 0}
            return call(request_obj)
        raise

def load_hf_model(name, cfg):
    assert name in TOKENIZERS, f"Tokenizer chưa được tải/audit: {name}. Chạy cell 3 trước."
    import torch
    from transformers import AutoModelForCausalLM
    gpu = torch.cuda.is_available()
    is_7b = name in ("SQLCoder-7B-2", "NSQL-Llama-2-7B")
    if is_7b and not gpu:
        raise RuntimeError("7B fp32 CPU không phù hợp: cần GPU >=16 GiB hoặc CUDA 4-bit được kiểm chứng")
    if is_7b and gpu and not USE_4BIT_7B and torch.cuda.get_device_properties(0).total_memory < 15 * 1024**3:
        raise RuntimeError("GPU <15 GiB, 7B fp16 dễ OOM; chạy trên GPU lớn hơn hoặc bật USE_4BIT_7B")
    kwargs = {"trust_remote_code": False, "low_cpu_mem_usage": True}
    if gpu:
        kwargs["device_map"] = "auto"
        if is_7b and USE_4BIT_7B:
            assert importlib.util.find_spec("bitsandbytes"), "Thiếu bitsandbytes (pip install bitsandbytes)"
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4")
        else:
            kwargs["torch_dtype"] = torch.float16
    else:
        kwargs["torch_dtype"] = torch.float32
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], **kwargs)
    model.eval()
    tok = TOKENIZERS[name]
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    return model, tok

def hf_generate(name, prompt, cfg, model, tok):
    import torch
    inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
    tokens = int(inputs["input_ids"].shape[-1])
    if tokens + cfg["max_new"] > cfg["context"]:
        raise ValueError(f"Token budget exceeded: {name}: {tokens}+{cfg['max_new']}>{cfg['context']}")
    inputs = {key: val.to(model.get_input_embeddings().weight.device) for key, val in inputs.items()}
    gen = {"max_new_tokens": cfg["max_new"], "do_sample": False,
           "eos_token_id": tok.eos_token_id, "pad_token_id": tok.pad_token_id}
    if name == "SQLCoder-7B-2": gen["num_beams"] = 4  # Model card Defog
    with torch.inference_mode():
        output = model.generate(**inputs, **gen)
    tail = output[0, tokens:]
    completion = tok.decode(tail, skip_special_tokens=True)
    # NSQL nhận SELECT là prefix input, phần decode chỉ là continuation.
    answer = (("SELECT" + completion) if name.startswith("NSQL") and
              not completion.lstrip().upper().startswith(("SELECT", "WITH"))
              else completion)
    return answer.strip(), tokens, int(tail.shape[0])

def validate_existing(name, run_id, spec, prompts_df):
    path = result_path(name, run_id)
    if not path.is_file(): return []
    old = pd.read_csv(path, keep_default_na=False)
    assert {"ID", "Model", "Status", "Generated Query"}.issubset(old.columns), f"Checkpoint sai định dạng: {path}"
    assert old["ID"].astype(int).is_unique
    assert set(old["ID"].astype(int)).issubset(set(prompts_df["ID"]))
    assert old["Model"].eq(name).all(), "Checkpoint model không khớp"
    hashes = dict(zip(prompts_df["ID"].astype(int), prompts_df["Prompt SHA256"]))
    if "Prompt SHA256" in old and old["Prompt SHA256"].astype(str).str.strip().ne("").all():
        for _, r in old.iterrows():
            assert r["Prompt SHA256"] == hashes[int(r["ID"])], (
                f"Prompt thay đổi, không được reuse checkpoint cũ: {path}")
    else:
        assert name == "Qwen2.5-Coder-3B" and REUSE_LEGACY_QWEN, (
            f"Checkpoint cũ thiếu Prompt SHA256: {path}. Di chuyển/đổi tên trước khi chạy lại.")
        print(f"Reusing legacy Qwen checkpoint: {path.name} (original full-DDL prompt)")
    return old.to_dict("records")

run_status = []
if RUN_INFERENCE:
    for name in RUN_MODELS:
        spec = MODEL_REGISTRY[name]
        group = model_prompts[model_prompts["Model"].eq(name)].sort_values("ID")
        selected = group if TEST_LIMIT is None else group.head(int(TEST_LIMIT))
        pending_by_run = {}
        for run_id in RUN_IDS:
            records = validate_existing(name, run_id, spec, group)
            complete_ids = {int(x["ID"]) for x in records if x["Status"] == "GENERATED"
                            and str(x["Generated Query"]).strip()}
            pending = selected[~selected["ID"].isin(complete_ids)]
            pending_by_run[run_id] = (records, pending)
        if all(len(pending) == 0 for _, pending in pending_by_run.values()):
            print(name, "— reused all selected predictions")
            continue
        model = tok = None
        try:
            if spec["backend"] in ("ollama", "ollama_gguf"):
                ensure_ollama_model(spec)
            else:
                if not importlib.util.find_spec("transformers") or not importlib.util.find_spec("torch"):
                    raise RuntimeError("Thiếu torch hoặc transformers")
                audit = TOKEN_AUDIT[TOKEN_AUDIT["Model"].eq(name)] if not TOKEN_AUDIT.empty else pd.DataFrame()
                if len(audit) != 43 or not audit["Fits Context"].all():
                    raise RuntimeError("Token audit chưa đủ 43 câu hoặc có prompt vượt context. Sửa Cell 2 trước.")
                model, tok = load_hf_model(name, spec)
            for run_id, (records, pending) in pending_by_run.items():
                checkpoint = result_path(name, run_id)
                for _, r in pending.iterrows():
                    t0 = time.perf_counter()
                    rec = {"ID": int(r["ID"]), "Inquiry URI": r["Inquiry URI"], "Question": r["Question"],
                           "Method": "SQL", "Model": name, "Run": run_id,
                           "Model ID": spec["model_id"], "Backend": spec["backend"],
                           "Schema Variant": spec["schema"], "Prompt SHA256": r["Prompt SHA256"],
                           "Serving Model": spec.get("serving_id", spec["model_id"]),
                           "Quantization": spec.get("quantization", "none"),
                           "Generated Query": "", "Prompt Tokens": "", "Output Tokens": "",
                           "Time (s)": "", "Status": "ERROR", "Error": ""}
                    try:
                        if spec["backend"] in ("ollama", "ollama_gguf"):
                            ans, pt, ot = ollama_generate(r["Prompt"], spec)
                        else:
                            ans, pt, ot = hf_generate(name, r["Prompt"], spec, model, tok)
                        if spec["backend"] == "ollama_gguf" and name.startswith("NSQL") and not ans.lstrip().upper().startswith(("SELECT", "WITH")):
                            ans = "SELECT" + ans
                        rec.update({"Generated Query": ans, "Prompt Tokens": pt, "Output Tokens": ot,
                                    "Status": "GENERATED" if ans else "EMPTY"})
                    except Exception as e:
                        rec["Error"] = f"{type(e).__name__}: {e}"
                    rec["Time (s)"] = round(time.perf_counter() - t0, 3)
                    records = [old for old in records if int(old["ID"]) != int(r["ID"])] + [rec]
                    save_checkpoint(checkpoint, records)
                    print(f"{name} run {run_id} Q{rec['ID']:02d}: {rec['Status']} ({rec['Time (s)']}s)", flush=True)
                run_status.append((name, run_id, checkpoint.name))
        except Exception as e:
            print(f"MODEL BLOCKED — {name}: {type(e).__name__}: {e}")
            if STOP_ON_MODEL_ERROR: raise
        finally:
            del model, tok
            gc.collect()
            if importlib.util.find_spec("torch"):
                import torch
                if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Inference pass finished. CSV files are resumable; only GENERATED rows are skipped.")
    if TEST_LIMIT is None:
        incomplete = []
        for model_name in RUN_MODELS:
            for run_id in RUN_IDS:
                f = result_path(model_name, run_id)
                if not f.is_file():
                    incomplete.append(f"{model_name}/run{run_id}: missing")
                    continue
                v = pd.read_csv(f, keep_default_na=False)
                complete_ids = {int(r["ID"]) for _, r in v.iterrows()
                                if r["Status"] == "GENERATED" and str(r["Generated Query"]).strip()}
                if complete_ids != set(questions["ID"].astype(int)):
                    incomplete.append(f"{model_name}/run{run_id}: {len(complete_ids)}/43 GENERATED")
        if incomplete:
            raise RuntimeError("Chưa chạy đủ 4 model: " + "; ".join(incomplete) + ". "
                               "Các checkpoint hoàn thành vẫn được lưu; chạy lại Cell 4 để resume.")
        print("ALL REQUESTED MODELS COMPLETE: 43 GENERATED / model / run.")
else:
    print("RUN_INFERENCE=False: không chạy model, chỉ dùng dữ liệu checkpoint hiện có ở Cell 5.")

Reusing legacy Qwen checkpoint: qwen2_5_coder_3b_run_01.csv (original full-DDL prompt)
Qwen2.5-Coder-3B — reused all selected predictions
NSQL-350M — reused all selected predictions
success 
Downloaded: hf.co/QuantFactory/sqlcoder-7b-2-GGUF:Q4_K_M
SQLCoder-7B-2 run 1 Q01: GENERATED (127.839s)
SQLCoder-7B-2 run 1 Q02: GENERATED (40.812s)
SQLCoder-7B-2 run 1 Q03: GENERATED (176.779s)
SQLCoder-7B-2 run 1 Q04: GENERATED (69.331s)
SQLCoder-7B-2 run 1 Q05: GENERATED (68.693s)
SQLCoder-7B-2 run 1 Q06: GENERATED (44.663s)
SQLCoder-7B-2 run 1 Q07: GENERATED (67.206s)
SQLCoder-7B-2 run 1 Q08: GENERATED (47.067s)
SQLCoder-7B-2 run 1 Q09: GENERATED (128.106s)
SQLCoder-7B-2 run 1 Q10: GENERATED (59.669s)
SQLCoder-7B-2 run 1 Q11: GENERATED (58.511s)
SQLCoder-7B-2 run 1 Q12: GENERATED (47.1s)
SQLCoder-7B-2 run 1 Q13: GENERATED (66.936s)
SQLCoder-7B-2 run 1 Q14: GENERATED (57.399s)
SQLCoder-7B-2 run 1 Q15: GENERATED (77.935s)
SQLCoder-7B-2 run 1 Q16: GENERATED (58.928s)
SQLCoder-7B-2 run 1 Q17: GENERA

In [12]:
import ast
from collections import Counter
from decimal import Decimal

# 1. Dùng ĐÚNG normalize/same_result từ 05_evaluation.ipynb để tránh đổi metric.
assert DB_FILE.is_file(), f"Không thấy DB: {DB_FILE}"
assert REF_MAP.is_file() and REF_DIR.is_dir(), "Thiếu SQL reference results"
assert EVAL_NOTEBOOK.is_file(), f"Thiếu notebook evaluation gốc: {EVAL_NOTEBOOK}"
assert importlib.util.find_spec("duckdb"), "Cần pip install duckdb để đánh giá SQL"
import duckdb
source = json.loads(EVAL_NOTEBOOK.read_text(encoding="utf-8"))
funcs = {}
for cell in source["cells"]:
    if cell["cell_type"] != "code": continue
    try: tree = ast.parse("".join(cell["source"]))
    except SyntaxError: continue  # bỏ qua Jupyter magic cells không phải Python
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in ("normalize", "same_result"):
            funcs[node.name] = node
assert set(funcs) == {"normalize", "same_result"}, "Không tìm thấy hai hàm so sánh gốc"
module = ast.fix_missing_locations(ast.Module(body=[funcs["normalize"], funcs["same_result"]], type_ignores=[]))
namespace = {"pd": pd, "re": re, "Decimal": Decimal, "Counter": Counter}
exec(compile(module, "<project_05_evaluation_functions>", "exec"), namespace)
same_result = namespace["same_result"]

# 2. ID → reference CSV theo mapping benchmark gốc.
reference = pd.read_csv(REF_MAP, keep_default_na=False)
assert {"ID", "SQL Reference"}.issubset(reference.columns)
reference["ID"] = reference["ID"].astype(int)
assert reference["ID"].is_unique
sql_refs = {int(r["ID"]): REF_DIR / (str(r["SQL Reference"]).rstrip("/").split("/")[-1] + ".csv")
            for _, r in reference.iterrows() if int(r["ID"]) in set(questions["ID"])}
assert set(sql_refs) == set(questions["ID"]) and all(p.is_file() for p in sql_refs.values())

# 3. Chỉ chuẩn hóa wrapper đầu ra, không sửa logic query, tên bảng/cột hoặc JOIN.
def extract_sql(text):
    text = str(text).strip()
    fenced = re.fullmatch(r"```(?:sql)?\s*\n(.*?)\n\s*```", text, flags=re.I | re.S)
    if fenced: text = fenced.group(1).strip()
    if text.startswith("[SQL]"):
        text = text[len("[SQL]"):].strip()
    if "[/SQL]" in text:
        text = text.split("[/SQL]", 1)[0].strip()
    return text

def error_type(err):
    s = str(err).lower()
    for needle, label in (("binder error", "Binder Error"), ("catalog error", "Catalog Error"),
                          ("parser error", "Parser Error"), ("conversion error", "Conversion Error"),
                          ("not a read-only", "Unsafe SQL"), ("multiple statements", "Unsafe SQL"),
                          ("timeout", "Timeout")):
        if needle in s: return label
    return "Other"

EVAL_OUT = OUT_DIR / "evaluation"
EVAL_OUT.mkdir(parents=True, exist_ok=True)
rows = []
question_ids = list(questions["ID"].astype(int))

for model_name in RUN_MODELS:
    for run_id in RUN_IDS:
        file = result_path(model_name, run_id)
        if not file.is_file():
            print("MISSING CHECKPOINT:", file.name)
            continue
        generation = pd.read_csv(file, keep_default_na=False)
        assert {"ID", "Status", "Generated Query"}.issubset(generation.columns)
        generation["ID"] = generation["ID"].astype(int)
        assert generation["ID"].is_unique and set(generation["ID"]).issubset(set(question_ids))
        by_id = generation.set_index("ID")
        run_dir = EVAL_OUT / MODEL_REGISTRY[model_name]["slug"] / f"run_{run_id:02d}"
        run_dir.mkdir(parents=True, exist_ok=True)
        con = duckdb.connect(str(DB_FILE), read_only=True)
        try:
            for qid in question_ids:
                record = by_id.loc[qid] if qid in by_id.index else None
                gen_status = str(record["Status"]) if record is not None else "MISSING"
                raw = str(record["Generated Query"]) if record is not None else ""
                query = extract_sql(raw) if gen_status == "GENERATED" else ""
                status, err, correct = "SKIPPED", "", 0
                outfile = run_dir / f"{qid:02d}.csv"
                outfile.unlink(missing_ok=True)
                t0 = time.perf_counter()
                if gen_status == "GENERATED":
                    try:
                        if not re.match(r"^(?:SELECT|WITH)\b", query, flags=re.I):
                            raise ValueError("Not a read-only SELECT/WITH query")
                        if ";" in query.rstrip().rstrip(";"):
                            raise ValueError("Multiple statements not permitted")
                        # Không chấp nhận các thao tác thay đổi DB trong truy vấn do model sinh.
                        if re.search(r"\b(?:INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|ATTACH|COPY|EXPORT|INSTALL|LOAD)\b", query, flags=re.I):
                            raise ValueError("Not a read-only SELECT/WITH query")
                        res = con.execute(query).fetchdf()
                        res.to_csv(outfile, index=False)
                        correct = int(same_result(sql_refs[qid], outfile))
                        status = "OK"
                    except Exception as e:
                        status, err = "ERROR", f"{type(e).__name__}: {e}"
                        outfile.unlink(missing_ok=True)
                rows.append({"Model": model_name, "Run": run_id, "ID": qid,
                             "Generation Status": gen_status, "Executed Query": query,
                             "Execution Status": status, "Correct": correct,
                             "Error Type": error_type(err) if err else "", "Error": err,
                             "Execution Time (s)": round(time.perf_counter() - t0, 3)})
        finally: con.close()

if not rows:
    print("Chưa có checkpoint inference; không tự tạo benchmark results.")
else:
    detail = pd.DataFrame(rows)
    detail.to_csv(EVAL_OUT / "baseline_evaluation_detail.csv", index=False, encoding="utf-8-sig")
    runs = (detail.groupby(["Model", "Run"], as_index=False)
            .agg(Questions=("ID", "size"), Generated=("Generation Status", lambda s: (s == "GENERATED").sum()),
                 Executed=("Execution Status", lambda s: (s == "OK").sum()),
                 Correct=("Correct", "sum")))
    runs["Complete"] = runs["Questions"].eq(43) & runs["Generated"].eq(43)
    runs["EA (%)"] = ((runs["Correct"] / 43 * 100).round(2)).where(runs["Complete"])
    runs.to_csv(EVAL_OUT / "baseline_ea_by_run.csv", index=False, encoding="utf-8-sig")
    print("=== EXECUTION ACCURACY (chỉ chốt EA khi đủ 43 SQL đã sinh) ===")
    display(runs)
    complete = runs[runs["Complete"]]
    if not complete.empty:
        model_summary = (complete.groupby("Model", as_index=False)
                         .agg(Complete_Runs=("Run", "nunique"), Mean_EA=("EA (%)", "mean"),
                              SD_EA=("EA (%)", "std")))
        model_summary.to_csv(EVAL_OUT / "baseline_summary.csv", index=False, encoding="utf-8-sig")
        display(model_summary)
        print("LƯU Ý: Mean_EA là trung bình các run baseline đã hoàn tất; không đồng nhất với AOEA của paper.")
    errors = detail[detail["Execution Status"].eq("ERROR")]
    if not errors.empty:
        display(pd.crosstab(errors["Model"], errors["Error Type"]))
    if QUADRANT_FILE.is_file():
        quadrant = pd.read_csv(QUADRANT_FILE, keep_default_na=False)
        quadrant = quadrant[quadrant["Method"].eq("SQL")][["ID", "Quadrant"]].copy()
        quadrant["ID"] = quadrant["ID"].astype(int)
        assert quadrant["ID"].is_unique
        if set(question_ids).issubset(set(quadrant["ID"])):
            qtab = detail.merge(quadrant, on="ID", validate="many_to_one")
            qtab.groupby(["Model", "Run", "Quadrant"], as_index=False)["Correct"].agg(["sum", "count"]).to_csv(
                EVAL_OUT / "baseline_by_quadrant.csv", index=False, encoding="utf-8-sig")
    print("Saved evaluation to:", EVAL_OUT)

=== EXECUTION ACCURACY (chỉ chốt EA khi đủ 43 SQL đã sinh) ===


,Model,Run,Questions,Generated,Executed,Correct,Complete,EA (%)
0,NSQL-350M,1,43,43,5,3,True,6.98
1,NSQL-Llama-2-7B,1,43,43,2,0,True,0.00
2,Qwen2.5-Coder-3B,1,43,43,11,4,True,9.30
3,SQLCoder-7B-2,1,43,43,12,4,True,9.30


,Model,Complete_Runs,Mean_EA,SD_EA
0,NSQL-350M,1,6.98,NaN
1,NSQL-Llama-2-7B,1,0.00,NaN
2,Qwen2.5-Coder-3B,1,9.30,NaN
3,SQLCoder-7B-2,1,9.30,NaN


LƯU Ý: Mean_EA là trung bình các run baseline đã hoàn tất; không đồng nhất với AOEA của paper.


Error Type,Binder Error,Catalog Error,Parser Error,Unsafe SQL
Model,,,,
NSQL-350M,32,3,3,0
NSQL-Llama-2-7B,2,0,1,38
Qwen2.5-Coder-3B,27,5,0,0
SQLCoder-7B-2,24,7,0,0


Saved evaluation to: C:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\related_methods\model_baselines\evaluation


In [13]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, OWL, RDFS

# Tìm các file RDF trong dự án
folders = [
    PROJECT_DIR / "data",
    PROJECT_DIR / "results/ontop",
    PROJECT_DIR / "results/ontology",
    PROJECT_DIR / "ontology"
]

files = sorted(set(
    p for folder in folders if folder.exists()
    for p in folder.rglob("*")
    if p.suffix.lower() in {".ttl", ".owl", ".rdf"}
))

RR = Namespace("http://www.w3.org/ns/r2rml#")

ontologies = []
mappings = []

# Phân loại theo nội dung RDF
for path in files:
    try:
        graph = Graph()
        graph.parse(path)
    except Exception:
        continue

    n_classes = len(set(graph.subjects(RDF.type, OWL.Class))) + len(
        set(graph.subjects(RDF.type, RDFS.Class))
    )
    n_maps = len(set(graph.subjects(RDF.type, RR.TriplesMap)))

    if n_classes:
        ontologies.append((path, graph))
    if n_maps:
        mappings.append((path, graph))

print("Ontology:", [str(p) for p, _ in ontologies])
print("R2RML:", [str(p) for p, _ in mappings])

assert len(ontologies) == 1, "Cần xác định đúng file ontology"
assert len(mappings) == 1, "Cần xác định đúng file mapping"

ontology_file, ontology = ontologies[0]
mapping_file, mapping = mappings[0]

print("Ontology triples:", len(ontology))
print("Mapping triples:", len(mapping))
print("R2RML Triples Maps:", len(set(
    mapping.subjects(RDF.type, RR.TriplesMap)
)))


Ontology: ['C:\\Users\\Win 11 New\\Desktop\\thuc-nghiem-co-so-httt\\data\\ACME_Insurance\\ontology\\insurance.ttl']
R2RML: ['C:\\Users\\Win 11 New\\Desktop\\thuc-nghiem-co-so-httt\\results\\ontop\\acme_mapping.ttl']
Ontology triples: 164
Mapping triples: 561
R2RML Triples Maps: 62
